<a href="https://colab.research.google.com/github/yuniecorn-dev/esaa_assignment2/blob/main/ESAA_OB_WEEK1_study_0904.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##텍스트 분류 실습 - 20 뉴스그룹 분류

- 텍스트 분류
  - 특정 문서의 분류를 학습 데이터를 통해 학습해
  - 모델을 생성한 뒤
  - 이 학습 모델을 이용해 다른 문서의 분류를 예측하는 것
- 사이킷런의 `fetch_20newsgroups()`를 활용해 뉴스그룹 텍스트 분류를 수행함.
- 텍스트를 정규화하고 **카운트·TF-IDF 방식으로 벡터화**한 뒤 **로지스틱 회귀**로 분류함.
- `GridSearchCV`와 `Pipeline`을 활용해 벡터화 및 하이퍼파라미터를 함께 튜닝하고 성능을 비교함.

###텍스트 정규화

In [1]:
from sklearn.datasets import fetch_20newsgroups

news_data = fetch_20newsgroups(subset='all', random_state=156)

In [2]:
print(news_data.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


In [3]:
import pandas as pd

print('target 클래스의 값과 분포도 \n', pd.Series(news_data.target).value_counts().sort_index())
print('target 클래스의 이름들 \n', news_data.target_names)

target 클래스의 값과 분포도 
 0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64
target 클래스의 이름들 
 ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


- 뉴스그룹 데이터에서 제목·작성자·소속·이메일 등의 헤더와 푸터 정보를 제거함.
- 이러한 정보는 Target 클래스와 유사해 높은 예측 성능을 만들 수 있으므로 순수한 기사 내용만 사용함.
- `remove` 파라미터로 불필요한 정보를 제거하고, `subset` 파라미터로 학습·테스트 데이터를 분리함.

In [4]:
from sklearn.datasets import fetch_20newsgroups

# subset='train'으로 학습용 데이터만 추출, remove=('headers', 'footers', 'quotes')로 내용만 추출
train_news = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'),
                                random_state=156)
X_train = train_news.data
y_train = train_news.target

# subset='test'으로 테스트 데이터만 추출, remove=('headers', 'footers', 'quotes')로 내용만 추출
test_news = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'),
                               random_state=156)
X_test = test_news.data
y_test = test_news.target
print('학습 데이터 크기 {0}, 테스트 데이터 크기 {1}'.format(len(train_news.data),
                                             len(test_news.data)))

학습 데이터 크기 11314, 테스트 데이터 크기 7532


###피처 벡터화 변환과 머신러닝 모델 학습/예측/평가

- 학습 데이터 11,314개와 테스트 데이터 7,532개를 `CountVectorizer`로 벡터화함.
- 테스트 데이터는 학습 데이터로 `fit()`한 동일한 `CountVectorizer` 객체의 `transform()`만 사용해야 함.
- 테스트 데이터에 `fit_transform()`을 사용하면 피처 개수가 달라져 학습된 모델로 예측할 수 없으므로 주의해야 함.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

# Count Vectorization으로 피처 벡터화 변환 수행.
cnt_vect = CountVectorizer()
cnt_vect.fit(X_train)
X_train_cnt_vect = cnt_vect.transform(X_train)

# 학습 데이터로 fit()된 CountVectorizer를 이용해 테스트 데이터를 피처 벡터화 변환 수행.
X_test_cnt_vect = cnt_vect.transform(X_test)

print('학습 데이터 텍스트의 CountVectorizer Shape:', X_train_cnt_vect.shape)

학습 데이터 텍스트의 CountVectorizer Shape: (11314, 101631)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# LogisticRegression을 이용하여 학습/예측/평가 수행.
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt_vect, y_train)
pred = lr_clf.predict(X_test_cnt_vect)
print('CountVectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

CountVectorized Logistic Regression의 예측 정확도는 0.617


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 벡터화를 적용해 학습 데이터 세트와 테스트 데이터 세트 변환.
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

# LogisticRegression을 이용해 학습/예측/평가 수행.
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

TF-IDF Logistic Regression의 예측 정확도는 0.678


- TF-IDF 벡터화가 단순 Count 기반 벡터화보다 높은 예측 정확도를 제공하며, 텍스트가 많은 분석에서 효과적임.
- 텍스트 정규화와 피처 벡터화 같은 전처리 방식 및 적절한 ML 알고리즘 선택이 성능 향상에 큰 영향을 줌.
- `TfidfVectorizer`의 `stop_words='english'`, `ngram_range=(1,2)`, `max_df=300` 등의 파라미터를 조정해 성능을 개선함.

In [8]:
# stop words 필터링을 추가하고 ngram을 기본 (1,1)에서 (1,2)로 변경해 피처 벡터화 적용.
tfidf_vect = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300)
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(
    accuracy_score(y_test, pred)))

TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.690


In [9]:
from sklearn.model_selection import GridSearchCV

# 최적 C값 도출 튜닝 수행. CV는 3폴드 세트로 설정.
params = {'C':[0.01, 0.1, 1, 4, 10]}
grid_cv_lr = GridSearchCV(lr_clf, param_grid=params, cv=3, scoring='accuracy', verbose=1)
grid_cv_lr.fit(X_train_tfidf_vect, y_train)
print('Logistic Regression best C parameter :', grid_cv_lr.best_params_)

# 최적 C값으로 학습된 grid_cv로 예측 및 정확도 평가
pred = grid_cv_lr.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(
    accuracy_score(y_test, pred)))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Logistic Regression best C parameter : {'C': 10}
TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.704


###사이킷런 파이프라인(Pipeline) 사용 및 GridSearchCV와의 결합

- 사이킷런의 `Pipeline`을 사용하면 데이터 전처리와 머신러닝 학습·예측 과정을 하나로 연결해 처리함.
- 전처리 결과를 별도로 저장하지 않고 스트림 방식으로 모델에 전달해 코드가 간결해지고 수행 시간을 절약함.
- 텍스트 벡터화뿐만 아니라 스케일링, 정규화, PCA 등의 전처리와 분류·회귀 모델도 함께 결합할 수 있음.

In [10]:
import os
from sklearn.pipeline import Pipeline

In [11]:
pipeline = Pipeline([('tfidf_vect', TfidfVectorizer(stop_words='english')),
                     'lr_clf', LogisticRegression(random_state=156)])

- `TfidfVectorizer`와 `LogisticRegression` 객체를 생성하고 Pipeline으로 연결함.
- Pipeline의 `fit()`과 `predict()`만으로 벡터화와 모델 학습·예측 과정을 한 번에 수행함.
- 전처리와 머신러닝 과정을 통일해 코드가 직관적이고 간결해짐.

In [12]:
from sklearn.pipeline import Pipeline

# TfidfVectorizer 객체를 tfidf_vect로, LogisticRegression 객체를 lr_clf로 생성하는 Pipeline 생성
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300)),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])

# 별도의 TfidfVectorizer 객체의 fit(), transform()과 LogisticRegression의 fit(), predict()가 필요없음
# pipeline의 fit()과 predict()만으로 한꺼번에 피처 벡터화와 ML 학습/예측이 가능
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(
    accuracy_score(y_test, pred)))

Pipeline을 통한 Logistic Regression의 예측 정확도는 0.704


- `Pipeline`과 `GridSearchCV`를 결합하면 TF-IDF 벡터화 파라미터와 Logistic Regression 하이퍼파라미터를 함께 최적화할 수 있음.
- `param_grid`의 키는 `객체명__파라미터명` 형식으로 지정하며, 예를 들어 `tfidf_vect__ngram_range`처럼 설정함.
- 튜닝할 파라미터가 많아지면 경우의 수와 학습 시간이 급증하므로 필요한 파라미터만 선별해 최적화해야 함.

In [63]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', min_df=3)),
    ('lr_clf', LogisticRegression(solver='liblinear')) # 텍스트 분류에 빠른 solver
])

params = {
    'tfidf_vect__ngram_range': [(1,1), (1,2)],
    'tfidf_vect__max_df': [100, 300, 700],
    'lr_clf__C': [1, 5, 10]
}

grid_cv_pipe = GridSearchCV(pipeline, param_grid=params, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)
grid_cv_pipe.fit(X_train['review'], y_train)
print(grid_cv_pipe.best_params_, grid_cv_pipe.best_score_)

pred = grid_cv_pipe.predict(X_test['review'])
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Fitting 3 folds for each of 18 candidates, totalling 54 fits
{'lr_clf__C': 10, 'tfidf_vect__max_df': 700, 'tfidf_vect__ngram_range': (1, 2)} 0.8828001298394387
Pipeline을 통한 Logistic Regression의 예측 정확도는 0.889


- `max_df=700`, `ngram_range=(1,2)`, `C=10` 조합에서 가장 좋은 검증 성능을 보였음.
- 최적화된 모델의 테스트 정확도는 약 0.702로 큰 폭의 성능 향상은 없었음.
- 희소 행렬 기반 텍스트 분류에는 로지스틱 회귀 외에 SVM과 나이브 베이즈도 활용할 수 있음.

##8.5 감성 분석

- 감성 분석은 텍스트의 주관적인 감정·의견·기분 등을 파악해 긍정 또는 부정 감성을 판단하는 방법임.
- 머신러닝 관점에서 학습 데이터와 레이블을 사용하는 지도학습 방식으로 감성을 예측할 수 있음.
- 비지도학습은 감성 어휘 사전인 Lexicon을 활용해 문서의 긍정·부정 감성을 판단함.

###지도학습 기반 감성 분석 실습 - IMDB 영화평

In [16]:
import pandas as pd

review_df = pd.read_csv('labeledTrainData.tsv', header=0, sep='\t', quoting=3)
review_df.head(3)

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."


In [17]:
print(review_df['review'][0])

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bit when it finally sta

- HTML에서 추출한 텍스트에 남아 있는 `<br />` 태그를 불필요한 피처이므로 공백으로 제거함.
- 판다스의 `str.replace()`를 이용해 문자열을 일괄적으로 처리함.
- 정규 표현식 `re.sub()`을 활용해 영어가 아닌 숫자·특수문자를 공백으로 변경해 텍스트를 정제함.

In [18]:
import re

# <br> html 태그는 replace 함수로 공백으로 변환
review_df['review'] = review_df['review'].str.replace('<br />', ' ')

# 파이썬의 정규 표현식 모듈인 re를 이용해 영어 문자열이 아닌 문자는 모두 공백으로 변환
review_df['review'] = review_df['review'].apply(lambda x: re.sub("[^a-zA-Z]", " ", x))

In [19]:
from sklearn.model_selection import train_test_split

class_df = review_df['sentiment']
feature_df = review_df.drop(['id', 'sentiment'], axis=1, inplace=False)

X_train, X_test, y_train, y_test = train_test_split(feature_df, class_df, test_size=0.3, random_state=156)
X_train.shape, X_test.shape

((17500, 1), (7500, 1))

- 감상평 텍스트를 Count와 TF-IDF 방식으로 각각 피처 벡터화한 뒤 LogisticRegression으로 분류함.
- `Pipeline`을 활용해 텍스트 벡터화와 머신러닝 학습·예측을 한 번에 수행함.
- 이진 분류 성능을 평가하기 위해 테스트 데이터의 정확도와 ROC-AUC를 함께 측정함.

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 스톱 워드는 English, ngram은 (1,2)로 설정해 CountVectorization 수행.
# LogisticRegression의 C는 10으로 설정
pipeline = Pipeline([
    ('cnt_vect', CountVectorizer(stop_words='english', ngram_range=(1,2))),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))])

# Pipeline 객체를 이용해 fit(), predict()로 학습/예측 수행. predict_proba()는 roc_auc 때문에 수행
pipeline.fit(X_train['review'], y_train)
pred = pipeline.predict(X_test['review'])
pred_probs = pipeline.predict_proba(X_test['review'])[:, 1]

print('예측 정확도는 {0:.4f}, ROC-AUC는 {1:.4f}'.format(accuracy_score(y_test, pred),
                                                 roc_auc_score(y_test, pred_probs)))

예측 정확도는 0.8861, ROC-AUC는 0.9503


In [25]:
# 스톱 워드는 english, filtering, ngram은 (1,2)로 설정해 TF-IDF 벡터화 수행
# LogisticRegression의 C는 10으로 설정
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))])

pipeline.fit(X_train['review'], y_train)
pred = pipeline.predict(X_test['review'])
pred_probs = pipeline.predict_proba(X_test['review'])[:,1]

print('예측 정확도 {0:.4f}, ROC-AUC는 {1:.4f}'.format(accuracy_score(y_test, pred),
                                                roc_auc_score(y_test, pred_probs)))

예측 정확도 0.8936, ROC-AUC는 0.9598


###비지도학습 기반 감성 분석 소개

- 비지도 감성 분석은 레이블이 없는 데이터에서 감성 어휘 사전인 Lexicon을 활용해 감성을 분석함.
- Lexicon은 단어별 긍정·부정 정도를 나타내는 감성 지수(Polarity Score)를 제공함.
- 대표적인 감성 사전 구현으로 NLTK가 있으며, 단어의 문맥·위치·POS 등을 고려해 감성을 판단함.
---
- NLTK의 WordNet은 단순한 영어 사전이 아니라 단어의 문맥상 의미인 시맨틱 정보를 제공하는 어휘 사전임.
- 같은 단어도 상황과 문맥에 따라 의미가 달라질 수 있으며, WordNet은 이러한 다양한 의미를 구분해 제공함.
- WordNet은 품사별 단어를 Synset으로 묶어 표현하며, Synset은 단어의 문맥과 의미 정보를 담는 핵심 개념임.
---
- NLTK 감성 사전은 유용하지만 예측 성능이 낮아 실제 업무에서는 SentiWordNet, VADER 등의 다른 사전을 주로 활용함.
- SentiWordNet은 긍정·부정·객관성 점수를 이용하고, VADER는 소셜 미디어 감성 분석에 특화되어 빠르고 높은 성능을 제공함.
- Pattern은 높은 성능이 장점이지만 Python 2.X만 지원하며, 이후 SentiWordNet과 VADER를 지도학습 방식과 비교함.

###SentiWordNet을 이용한 감성 분석

####WordNet Synset과 SentiWordNet SentiSynset 클래스의 이해

In [26]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [27]:
from nltk.corpus import wordnet as wn

term = 'present'

# 'present'라는 단어로 wordnet의 synsets 생성
synsets = wn.synsets(term)
print('synsets() 반환 type :', type(synsets))
print('synsets() 반환 값 개수 :', len(synsets))
print('synsets() 반환 값 : ', synsets)

synsets() 반환 type : <class 'list'>
synsets() 반환 값 개수 : 18
synsets() 반환 값 :  [Synset('present.n.01'), Synset('present.n.02'), Synset('present.n.03'), Synset('show.v.01'), Synset('present.v.02'), Synset('stage.v.01'), Synset('present.v.04'), Synset('present.v.05'), Synset('award.v.01'), Synset('give.v.08'), Synset('deliver.v.01'), Synset('introduce.v.01'), Synset('portray.v.04'), Synset('confront.v.03'), Synset('present.v.12'), Synset('salute.v.06'), Synset('present.a.01'), Synset('present.a.02')]


- `synsets()`는 하나의 단어가 가진 여러 의미를 나타내는 Synset 객체들의 리스트를 반환함.
- `present.n.01`에서 `present`는 단어, `n`은 명사 품사, `01`은 해당 품사의 여러 의미를 구분하는 인덱스임.
- Synset 객체는 품사(POS), 정의(Definition), 부명제(Lemma) 등 다양한 시맨틱 정보를 제공함.

In [29]:
for synset in synsets :
  print('##### Synset name : ', synset.name(), '#####')
  print('POS :', synset.lexname())
  print('Definition:', synset.definition())
  print('Lemas:', synset.lemma_names())

##### Synset name :  present.n.01 #####
POS : noun.time
Definition: the period of time that is happening now; any continuous stretch of time including the moment of speech
Lemas: ['present', 'nowadays']
##### Synset name :  present.n.02 #####
POS : noun.possession
Definition: something presented as a gift
Lemas: ['present']
##### Synset name :  present.n.03 #####
POS : noun.communication
Definition: a verb tense that expresses actions or states at the time of speaking
Lemas: ['present', 'present_tense']
##### Synset name :  show.v.01 #####
POS : verb.perception
Definition: give an exhibition of to an interested audience
Lemas: ['show', 'demo', 'exhibit', 'present', 'demonstrate']
##### Synset name :  present.v.02 #####
POS : verb.communication
Definition: bring forward and present to the mind
Lemas: ['present', 'represent', 'lay_out']
##### Synset name :  stage.v.01 #####
POS : verb.creation
Definition: perform (a play), especially on a stage
Lemas: ['stage', 'present', 'represent']
##

- 같은 명사라도 `present.n.01`은 ‘현재’, `present.n.02`는 ‘선물’처럼 서로 다른 의미를 나타냄.
- Synset은 하나의 단어가 가진 다양한 시맨틱 정보를 개별 객체로 표현하며, 품사와 정의 등을 함께 제공함.
- WordNet의 `path_similarity()` 메서드를 이용하면 `tree`, `lion`, `tiger`, `cat`, `dog` 등의 단어 간 의미적 유사도를 비교할 수 있음.

In [35]:
# synset 객체를 단어별로 생성합니다.
tree = wn.synset('tree.n.01')
lion = wn.synset('lion.n.01')
tiger = wn.synset('tiger.n.02')
cat = wn.synset('cat.n.01')
dog = wn.synset('dog.n.01')

entities = [tree, lion, tiger, cat, dog]
similarities = []
entity_names = [entity.name().split('.')[0] for entity in entities]

# 단어별 synset을 반복하면서 다른 단어의 synset과 유사도를 측정합니다.
for entity in entities:
  similarity = [round(entity.path_similarity(compared_entity), 2) for compared_entity in entities]
  similarities.append(similarity)

# 개별 단어별 synset과 다른 단어의 synset과의 유사도를 DataFrame 형태로 저장합니다.
similarity_df = pd.DataFrame(similarities, columns=entitiy_names, index=entity_names)
similarity_df

,tree,lion,tiger,cat,dog
tree,1.00,0.07,0.07,0.08,0.12
lion,0.07,1.00,0.33,0.25,0.17
tiger,0.07,0.33,1.00,0.25,0.17
cat,0.08,0.25,0.25,1.00,0.20
dog,0.12,0.17,0.17,0.20,1.00


In [36]:
import nltk
from nltk.corpus import sentiwordnet as swn

senti_synsets = list(swn.senti_synsets('slow'))
print('senti_synsets() 반환 type:', type(senti_synsets))
print('senti_synsets() 반환 값 개수:', len(senti_synsets))
print('senti_synsets() 반환 값:', senti_synsets)

senti_synsets() 반환 type: <class 'list'>
senti_synsets() 반환 값 개수: 11
senti_synsets() 반환 값: [SentiSynset('decelerate.v.01'), SentiSynset('slow.v.02'), SentiSynset('slow.v.03'), SentiSynset('slow.a.01'), SentiSynset('slow.a.02'), SentiSynset('dense.s.04'), SentiSynset('slow.a.04'), SentiSynset('boring.s.01'), SentiSynset('dull.s.08'), SentiSynset('slowly.r.01'), SentiSynset('behind.r.03')]


- `SentiSynset`은 단어의 긍정·부정 감성 지수와 객관성 지수를 제공함.
- 감성 지수는 긍정 감성 지수와 부정 감성 지수로 나뉘어 단어의 감정적 성향을 나타냄.
- 감성이 전혀 없는 단어는 객관성 지수가 1이고 긍정·부정 감성 지수는 모두 0이 됨.

In [37]:
import nltk
from nltk.corpus import sentiwordnet as swn

father = swn.senti_synset('father.n.01')
print('father 긍정감성 지수: ', father.pos_score())
print('father 부정감성 지수: ', father.neg_score())
print('father 객관성 지수: ', father.obj_score())
print('\n')
fabulous = swn.senti_synset('fabulous.a.01')
print('fabulous 긍정감성 지수: ', fabulous.pos_score())
print('fabulous 부정감성 지수: ', fabulous.neg_score())

father 긍정감성 지수:  0.0
father 부정감성 지수:  0.0
father 객관성 지수:  1.0


fabulous 긍정감성 지수:  0.875
fabulous 부정감성 지수:  0.125


####SentiWordNet을 이용한 영화 감상평 감성 분석
1 문서를 문장·단어 단위로 분해

2 문장을 단어 단위로 토큰화 하고 품사 태깅

3 품사 태깅된 단어로 Synset과 SentiSynset을 생성

4 긍정·부정 감성 지수를 계산. 감성 지수를 합산해 임계값과 비교하여 긍정 또는 부정 감성으로 최종 분류

In [47]:
from nltk.corpus import wordnet as wn

# 간단한 NTLK PennTreebank Tag를 기반으로 WordNet 기반의 품사 Tag로 변환
def penn_to_wn(tag):
  if tag.startswith('J'):
    return wn.ADJ
  elif tag.startswith('N'):
    return wn.NOUN
  elif tag.startswith('R'):
    return wn.ADV
  elif tag.startswith('V'):
    return wn.VERB

- 문장 → 단어 토큰 → 품사 태깅 후에 SentiSynset 클래스를 생성하고 Polarity Score를 합산하는 함수를 생성
- 총 감성 지수(긍정 감성 지수 + 부정 감성 지수)
  - 0 이상: 긍정 감성
  - 그렇지 않을 경우: 부정 감성

In [54]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn  # wn 모듈 추가
from nltk import sent_tokenize, word_tokenize, pos_tag

def swn_polarity(text):
    # 감성 지수 초기화
    sentiment = 0.0
    tokens_count = 0

    lemmatizer = WordNetLemmatizer()
    raw_sentences = sent_tokenize(text)

    # 분해된 문장별로 단어 토큰 -> 품사 태깅 후에 SentiSynset 생성 -> 감성 지수 합산
    for raw_sentence in raw_sentences:
        # NTLK 기반의 품사 태깅 문장 추출
        tagged_sentence = pos_tag(word_tokenize(raw_sentence))

        for word, tag in tagged_sentence:
            # WordNet 기반 품사 태깅과 어근 추출
            wn_tag = penn_to_wn(tag)
            if wn_tag not in (wn.NOUN, wn.ADJ, wn.ADV):
                continue

            lemma = lemmatizer.lemmatize(word, pos=wn_tag)
            if not lemma:
                continue

            # 어근을 추출한 단어와 WordNet 기반 품사 태깅을 입력해 Synset 객체를 생성
            synsets = wn.synsets(lemma, pos=wn_tag)
            if not synsets:
                continue
            synset = synsets[0]
            # sentiwordnet의 감성 단어 분석으로 감성 synset 추출
            # 모든 단어에 대해 긍정 감성 지수는 +로 부정 감성 지수는 -로 합산해 감성 지수 계산
            swn_synset = swn.senti_synset(synset.name())
            sentiment += (swn_synset.pos_score() - swn_synset.neg_score())
            tokens_count += 1

    if not tokens_count:
        return 0

    # 총 score가 0 이상일 경우 긍정(Positive) 1, 그렇지 않을 경우 부정(Negative) 0 반환
    if sentiment >= 0:
        return 1

    return 0

- `swn_polarity(text)` 함수를 IMDB 개별 감상평에 적용해 긍정·부정 감성을 예측함.
- 예측 결과를 `review_df`의 새로운 `preds` 칼럼에 저장하고 실제 `sentiment`와 비교해 정확도·정밀도·재현율을 측정함.

In [55]:
review_df['preds'] = review_df['review'].apply(lambda x : swn_polarity(x))
y_target = review_df['sentiment'].values
preds = review_df['preds'].values

In [56]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score
from sklearn.metrics import recall_score, f1_score, roc_auc_score
import numpy as np

print(confusion_matrix(y_target, preds))
print("정확도:", np.round(accuracy_score(y_target, preds), 4))
print("정밀도:", np.round(precision_score(y_target, preds), 4))
print("재현율:", np.round(recall_score(y_target, preds), 4))

[[7669 4831]
 [3635 8865]]
정확도: 0.6614
정밀도: 0.6473
재현율: 0.7092


###VADER를 이용한 감성 분석

- VADER는 소셜 미디어 감성 분석에 특화된 룰 기반 Lexicon으로 `SentimentIntensityAnalyzer` 클래스를 이용함.
- NLTK의 서브 모듈이나 별도 `vaderSentiment` 패키지로 설치해 사용할 수 있음.
- VADER는 버전에 따라 감성 분석 결과가 달라질 수 있으므로 설치된 버전을 확인할 필요가 있음.

In [57]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

senti_analyzer = SentimentIntensityAnalyzer()
senti_scores = senti_analyzer.polarity_scores(review_df['review'][0])
print(senti_scores)

{'neg': 0.13, 'neu': 0.743, 'pos': 0.127, 'compound': -0.7943}


- VADER는 `SentimentIntensityAnalyzer` 객체를 생성한 뒤 `polarity_scores()`를 호출해 문서의 감성 점수를 계산함.
- `polarity_scores()`는 `neg`, `neu`, `pos`, `compound`의 4가지 감성 점수를 딕셔너리 형태로 반환함.
- `compound`는 전체 감성을 -1\~1 사이의 값으로 나타내며, 일반적으로 0.1 이상이면 긍정, 이하면 부정으로 판단함.
- `vader_polarity()` 함수에 감상평과 임계값을 입력해 각 문서의 긍정·부정 감성을 판별함.
- `apply()`와 `lambda`를 이용해 모든 IMDB 감상평을 분석하고 결과를 `vader_preds`에 저장한 뒤 예측 성능을 평가함.

In [58]:
def vader_polarity(review, threshold=0.1):
  analyzer = SentimentIntensityAnalyzer()
  scores = analyzer.polarity_scores(review)

  # compound 값에 기반해 threshold 입력값보다 크면 1, 그렇지 않으면 0을 반환
  agg_score = scores['compound']
  final_sentiment = 1 if agg_score >= threshold else 0
  return final_sentiment

# apply lambda 식을 이용해 레코드별로 vader_polarity()를 수행하고 결과를 'vader_preds'에 저장
review_df['vader_preds'] = review_df['review'].apply(lambda x : vader_polarity(x, 0.1))
y_target = review_df['sentiment'].values
vader_preds = review_df['vader_preds'].values

print(confusion_matrix(y_target, vader_preds))
print("정확도:", np.round(accuracy_score(y_target, vader_preds), 4))
print("정밀도:", np.round(precision_score(y_target, vader_preds), 4))
print("재현율:", np.round(recall_score(y_target, vader_preds), 4))

[[ 6747  5753]
 [ 1858 10642]]
정확도: 0.6956
정밀도: 0.6491
재현율: 0.8514


- VADER는 SentiWordNet보다 정확도가 향상되었으며, 특히 재현율이 약 85.14%로 크게 높아짐.
- 성능 비교에서 SentiWordNet은 정확도 **0.6613**, 정밀도 **0.6472**, 재현율 **0.7091**을 기록함.
- VADER는 정확도 **0.6956**, 정밀도 **0.6491**, 재현율 **0.8514**로 전반적으로 더 높은 성능을 보임.
- 감성 사전 기반 분석은 지도학습보다 성능이 낮지만, 레이블이 없는 데이터에서도 감성 분석이 가능하다는 장점이 있음.
- Pattern 역시 뛰어난 감성 사전이지만 Python 3에서의 지원이 제한적이며, 감성 사전 방식은 레이블이 없는 상황에서 유용하게 활용할 수 있음.